# DS2002 · Capstone Workshop

**Lecture — 2026-11-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Capstone workshop

Only meeting this week — no class Wednesday or Friday for Thanksgiving. Use the hour on your own project.

Four patterns below, matching the four things teams asked about most last week. Take what you need and get back to your data.

Realistically you lose most of this week to the break, so the goal today is to leave with your cleaning finished and pushed. That way the week of the 30th is analysis rather than catching up.

### 1. Vendor ids in five formats

This one shows up in every team's data. Normalize once, in a function, before any join.

In [ ]:
import pandas as pd

s = pd.Series(['V-01', 'V_01', 'V01', 'v-01', 'V 01', 'v1'])

def normalize_vendor_id(series):
    """Everything to the canonical V-NN form."""
    stripped = (series.astype(str).str.upper()
                .str.replace(r'[^A-Z0-9]', '', regex=True))
    letters = stripped.str.extract(r'^([A-Z]+)', expand=False)
    digits = stripped.str.extract(r'([0-9]+)$', expand=False)
    return letters + '-' + digits.str.zfill(2)

out = normalize_vendor_id(s)
for before, after in zip(s, out):
    print(f'{before:6s} -> {after}')
print()
print('distinct before:', s.nunique(), '-> after:', out.nunique())

Note `zfill(2)`: without it, `v1` becomes `V-1` and stays separate from `V-01`. Padding is the difference between five vendors and one.

### 2. The weather join across several game dates

Six game weekends means six date ranges, or one range with the non-game days filtered out. One request is better than six — and cache it, because you will re-run this cell many times.

In [ ]:
game_dates = pd.to_datetime(['2026-09-05', '2026-09-19', '2026-10-10',
                             '2026-10-24', '2026-11-07', '2026-11-21'])

# Pretend this came back from the API across the full span
full_range = pd.DataFrame({
    'date': pd.date_range('2026-09-01', '2026-11-30'),
})
full_range['precip_mm'] = 0.0
full_range.loc[full_range['date'].isin(game_dates[[2, 5]]), 'precip_mm'] = [22.5, 8.0]

game_weather = full_range[full_range['date'].isin(game_dates)].copy()
print(game_weather)
assert len(game_weather) == len(game_dates), 'missing weather for a game date'
print('\nall', len(game_dates), 'game dates have weather.')

That assertion is the important line. If the API range does not fully cover your game dates, you want to know here rather than after a left join has quietly filled two games with `NaN` rain.

**Sales side:** your order timestamps need `.dt.normalize()` before they will match these dates. That is still the most common single bug in this project.

### 3. Question 5 — signal or artifact

The question is whether a poncho surge is real demand or an inventory effect. If a vendor sold 95 ponchos and had 95 in stock, you did not measure demand — you measured supply.

The pattern: join sales to the inventory table and look for anything that sold out.

In [ ]:
sales = pd.DataFrame({
    'vendor_id': ['V-18', 'V-18', 'V-01'],
    'date': pd.to_datetime(['2026-10-10', '2026-10-24', '2026-10-10']),
    'item': ['Rain Poncho', 'Rain Poncho', 'Cheeseburger'],
    'units_sold': [95, 40, 180],
})
inventory = pd.DataFrame({
    'vendor_id': ['V-18', 'V-18', 'V-01'],
    'date': pd.to_datetime(['2026-10-10', '2026-10-24', '2026-10-10']),
    'item': ['Rain Poncho', 'Rain Poncho', 'Cheeseburger'],
    'on_hand_start': [95, 120, 400],
})

check = sales.merge(inventory, on=['vendor_id', 'date', 'item'],
                    how='left', validate='one_to_one')
check['sold_out'] = check['units_sold'] >= check['on_hand_start']
check['pct_of_stock'] = (100 * check['units_sold'] / check['on_hand_start']).round(1)
print(check[['vendor_id', 'date', 'item', 'units_sold',
             'on_hand_start', 'pct_of_stock', 'sold_out']])

The October 10th ponchos sold out at exactly 95 of 95. That number is a **floor** on demand, not a measurement of it — real demand was higher and we cannot say how much. The 24th sold 40 of 120, which is a genuine demand figure.

This distinction is the whole of Question 5, and it changes your recommendation: "stock at least 95, and we do not know the ceiling" is honest, and it argues for stocking well above 95 next time.

### 4. Splitting work without merge conflicts

Three or four people and one notebook does not work; you learned that on the midterm. For the next two weeks:

- One notebook per person for exploration, named for the person.
- One shared notebook that only the named owner commits.
- Cleaning functions in one place. Everyone imports the same clean frame rather than cleaning it their own way.
- Pull before you start, push before you stop.

### Before break

- Cleaning finished and pushed
- Weather joined for all six game dates, with the assertion passing
- One person named as owner of the shared notebook over the break
- Everyone knows what they are picking up on the 30th

Have a good Thanksgiving.